In [ ]:
# Cell 1: Imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

# Cell 2: Scrape Billboard Hot 100
url = "https://www.billboard.com/charts/hot-100/"
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, 'html.parser')
    print("Successfully fetched Billboard page!")
else:
    print(f"Failed to fetch page. Status code: {response.status_code}")

# Cell 3: Parse the HTML
# Note: Billboard's CSS classes can change slightly, but the structure generally uses 
# reusable list item design elements for rows.
songs = []
artists = []

# Target the main list items containing the chart data
chart_rows = soup.select('ul.o-chart-results-list-row')

for row in chart_rows:
    # Extract Song Title (typically inside the first h3 nested under the list item)
    title_element = row.select_one('h3#title-of-a-story')
    
    # Extract Artist Name (typically the span immediately following or near the title)
    # We navigate to the parent container of the title to find the correct artist span
    if title_element:
        title = title_element.get_text(strip=True)
        
        # The artist is usually in the next sibling span element
        artist_element = title_element.find_next('span', class_='c-label')
        artist = artist_element.get_text(strip=True) if artist_element else "Unknown"
        
        songs.append(title.lower()) # Lowercase for easier user matching later
        artists.append(artist.lower())

# Cell 4: Create DataFrame and Clean
df_hot = pd.DataFrame({
    'song': songs,
    'artist': artists
})

# Quick data cleaning
df_hot = df_hot.drop_duplicates().dropna().reset_index(drop=True)
print(f"Collected {len(df_hot)} unique hot songs.")
print(df_hot.head())

# Cell 5: Export to CSV
os.makedirs('../data', exist_ok=True)
df_hot.to_csv('../data/hot_songs.csv', index=False)
print("Saved hot_songs.csv to data folder!")